In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
kmader_skin_cancer_mnist_ham10000_path = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')

print('Data source import complete.')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv')

# Check basic info
print(df.shape)          # Should be (10015, 7)
print(df.columns)        # lesion_id, image_id, dx, dx_type, age, sex, localization
print(df.isnull().sum()) # Check missing values in metadata itself
print(df['dx'].value_counts()) # Class distribution

(10015, 7)
Index(['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization'], dtype='object')
lesion_id        0
image_id         0
dx               0
dx_type          0
age             57
sex              0
localization     0
dtype: int64
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


# **Kaggle setup & imports**

In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Core imports
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# PyTorch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

PyTorch: 2.10.0+cu128
CUDA available: True
Device: Tesla T4


# **Locate HAM10000 files**

In [ ]:
# Standard Kaggle path when you add the HAM10000 dataset
BASE_DIR = '/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000'

# Verify the dataset is attached
print("Contents of base dir:")
print(os.listdir(BASE_DIR))

# CSV path
CSV_PATH = os.path.join(BASE_DIR, 'HAM10000_metadata.csv')

# Build image_id -> full path mapping across both image folders
image_dirs = [
    os.path.join(BASE_DIR, 'HAM10000_images_part_1'),
    os.path.join(BASE_DIR, 'HAM10000_images_part_2'),
]

image_path_map = {}
for d in image_dirs:
    if os.path.exists(d):
        for fname in os.listdir(d):
            if fname.endswith('.jpg'):
                image_id = fname.replace('.jpg', '')
                image_path_map[image_id] = os.path.join(d, fname)

print(f"\nTotal images found: {len(image_path_map)}")
# Expect: 10015

Contents of base dir:
['hmnist_8_8_RGB.csv', 'hmnist_28_28_RGB.csv', 'HAM10000_images_part_1', 'ham10000_images_part_1', 'hmnist_8_8_L.csv', 'HAM10000_images_part_2', 'ham10000_images_part_2', 'hmnist_28_28_L.csv', 'HAM10000_metadata.csv']

Total images found: 10015


# **Loading & cleaning the CSV**

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Total rows: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nMissing values per column:")
print(df.isnull().sum())

# Fill missing values
df['age'] = df['age'].fillna(df['age'].median())
df['sex'] = df['sex'].fillna('unknown')
df['localization'] = df['localization'].fillna('unknown')

# Confirm no missing values remain
assert df.isnull().sum().sum() == 0, "Still has missing values!"

# Attach image path
df['image_path'] = df['image_id'].map(image_path_map)
missing_paths = df['image_path'].isnull().sum()
print(f"\nRows with missing image files: {missing_paths}")
assert missing_paths == 0, "Some images couldn't be located!"

Total rows: 10015

Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization']

Missing values per column:
lesion_id        0
image_id         0
dx               0
dx_type          0
age             57
sex              0
localization     0
dtype: int64

Rows with missing image files: 0


# **Label encoding & class counts**

In [ ]:
# Map each disease string to a number
label_map = {
    'mel':   0,
    'nv':    1,
    'bcc':   2,
    'akiec': 3,
    'bkl':   4,
    'df':    5,
    'vasc':  6,
}
class_names = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']

# Add a 'label' column by looking up each row's dx
labels = []
for i in range(len(df)):
    dx_string = df.loc[i, 'dx']
    labels.append(label_map[dx_string])
df['label'] = labels

# Count how many of each class
print("Class distribution:")
for class_idx in range(7):
    count = 0
    for i in range(len(df)):
        if df.loc[i, 'label'] == class_idx:
            count += 1
    percent = 100 * count / len(df)
    print(f"  {class_names[class_idx]:6s} (label {class_idx}): {count:5d}  ({percent:5.2f}%)")

Class distribution:
  MEL    (label 0):  1113  (11.11%)
  NV     (label 1):  6705  (66.95%)
  BCC    (label 2):   514  ( 5.13%)
  AKIEC  (label 3):   327  ( 3.27%)
  BKL    (label 4):  1099  (10.97%)
  DF     (label 5):   115  ( 1.15%)
  VASC   (label 6):   142  ( 1.42%)


# **Metadata encoding (build the 19-dim vector)**

In [ ]:
# Step 1: Normalize age to [0, 1]
age_min = df['age'].min()
age_max = df['age'].max()
print(f"Age range: {age_min} to {age_max}")

age_norm_list = []
for i in range(len(df)):
    age = df.loc[i, 'age']
    age_normalized = (age - age_min) / (age_max - age_min)
    age_norm_list.append(age_normalized)
df['age_norm'] = age_norm_list

# Step 2: One-hot encode sex (3 categories)
sex_categories = ['male', 'female', 'unknown']

# Step 3: Find all localization categories
all_localizations = []
for i in range(len(df)):
    loc = df.loc[i, 'localization']
    if loc not in all_localizations:
        all_localizations.append(loc)
all_localizations.sort()
print(f"Found {len(all_localizations)} localization categories: {all_localizations}")

# Step 4: Build the metadata vector for each row
# Order: [age_norm, sex_male, sex_female, sex_unknown, loc_1, loc_2, ..., loc_N]
metadata_list = []
for i in range(len(df)):
    row_vector = []

    # Age (1 dim)
    row_vector.append(df.loc[i, 'age_norm'])

    # Sex one-hot (3 dims)
    current_sex = df.loc[i, 'sex']
    for s in sex_categories:
        if current_sex == s:
            row_vector.append(1.0)
        else:
            row_vector.append(0.0)

    # Localization one-hot (N dims)
    current_loc = df.loc[i, 'localization']
    for loc in all_localizations:
        if current_loc == loc:
            row_vector.append(1.0)
        else:
            row_vector.append(0.0)

    metadata_list.append(row_vector)

# Convert to numpy array
metadata_array = np.array(metadata_list, dtype=np.float32)

print(f"\nMetadata shape: {metadata_array.shape}")
META_DIM = metadata_array.shape[1]
print(f"META_DIM = {META_DIM}  (1 age + 3 sex + {len(all_localizations)} localization)")

Age range: 0.0 to 85.0
Found 15 localization categories: ['abdomen', 'acral', 'back', 'chest', 'ear', 'face', 'foot', 'genital', 'hand', 'lower extremity', 'neck', 'scalp', 'trunk', 'unknown', 'upper extremity']

Metadata shape: (10015, 19)
META_DIM = 19  (1 age + 3 sex + 15 localization)


# **Stratified train/val/test split (70/10/20)**

In [ ]:
# We still use sklearn for the actual split (writing stratified sampling
# from scratch is messy), but in a very plain way

# All row indices
all_indices = list(range(len(df)))
all_labels = df['label'].tolist()

# First split: separate test set (20%)
train_val_indices, test_indices = train_test_split(
    all_indices,
    test_size=0.20,
    stratify=all_labels,
    random_state=SEED,
)

# Get the labels for the remaining 80% so we can stratify again
train_val_labels = []
for idx in train_val_indices:
    train_val_labels.append(all_labels[idx])

# Second split: from the 80%, take 12.5% as val (= 10% of total)
train_indices, val_indices = train_test_split(
    train_val_indices,
    test_size=0.125,
    stratify=train_val_labels,
    random_state=SEED,
)

print(f"Train: {len(train_indices)}  ({100*len(train_indices)/len(df):.1f}%)")
print(f"Val:   {len(val_indices)}  ({100*len(val_indices)/len(df):.1f}%)")
print(f"Test:  {len(test_indices)}  ({100*len(test_indices)/len(df):.1f}%)")

# Build separate dataframes for each split
train_df = df.iloc[train_indices].reset_index(drop=True)
val_df   = df.iloc[val_indices].reset_index(drop=True)
test_df  = df.iloc[test_indices].reset_index(drop=True)

# Build separate metadata arrays for each split
train_meta = metadata_array[train_indices]
val_meta   = metadata_array[val_indices]
test_meta  = metadata_array[test_indices]

# Verify class proportions are preserved
print("\nClass proportions per split:")
print(f"{'Class':<8}{'Train':>10}{'Val':>10}{'Test':>10}")
for class_idx in range(7):
    # Count in train
    train_count = 0
    for i in range(len(train_df)):
        if train_df.loc[i, 'label'] == class_idx:
            train_count += 1
    train_prop = train_count / len(train_df)

    # Count in val
    val_count = 0
    for i in range(len(val_df)):
        if val_df.loc[i, 'label'] == class_idx:
            val_count += 1
    val_prop = val_count / len(val_df)

    # Count in test
    test_count = 0
    for i in range(len(test_df)):
        if test_df.loc[i, 'label'] == class_idx:
            test_count += 1
    test_prop = test_count / len(test_df)

    print(f"{class_names[class_idx]:<8}{train_prop:>10.4f}{val_prop:>10.4f}{test_prop:>10.4f}")

Train: 7010  (70.0%)
Val:   1002  (10.0%)
Test:  2003  (20.0%)

Class proportions per split:
Class        Train       Val      Test
MEL         0.1111    0.1108    0.1113
NV          0.6695    0.6697    0.6695
BCC         0.0514    0.0509    0.0514
AKIEC       0.0327    0.0329    0.0325
BKL         0.1097    0.1098    0.1098
DF          0.0114    0.0120    0.0115
VASC        0.0143    0.0140    0.0140


# **Class weights for weighted CrossEntropy**

In [ ]:
# Get the labels in the training set as a plain list
train_labels = train_df['label'].tolist()

# Use sklearn to compute balanced weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2, 3, 4, 5, 6]),
    y=np.array(train_labels),
)

# Convert to a PyTorch tensor (we'll pass this to CrossEntropyLoss later)
class_weights_tensor = torch.FloatTensor(class_weights)

print("Class weights (from training set):")
for class_idx in range(7):
    print(f"  {class_names[class_idx]:6s}: {class_weights[class_idx]:.4f}")

Class weights (from training set):
  MEL   : 1.2855
  NV    : 0.2134
  BCC   : 2.7817
  AKIEC : 4.3731
  BKL   : 1.3022
  DF    : 12.5179
  VASC  : 10.0143


# **Image transforms**

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# **The SkinLesionDataset class**

In [ ]:
class SkinLesionDataset(Dataset):
    """
    Loads (image, metadata, label) tuples.
    If modality_dropout_prob > 0, randomly zeroes the metadata vector.
    """

    def __init__(self, dataframe, metadata_array, transform, modality_dropout_prob):
        self.df = dataframe.reset_index(drop=True)
        self.metadata = metadata_array
        self.transform = transform
        self.dropout_prob = modality_dropout_prob

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Get the row
        row = self.df.iloc[idx]

        # Load and transform the image
        image_path = row['image_path']
        image = Image.open(image_path).convert('RGB')
        image = self.transform(image)

        # Get the metadata vector
        meta_numpy = self.metadata[idx]
        meta = torch.FloatTensor(meta_numpy)

        # Modality Dropout: maybe zero out the metadata
        if self.dropout_prob > 0:
            random_number = torch.rand(1).item()
            if random_number < self.dropout_prob:
                meta = torch.zeros_like(meta)

        # Get the label
        label_value = row['label']
        label = torch.tensor(label_value, dtype=torch.long)

        return image, meta, label

# **Build DataLoaders**

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2

# Training dataset: dropout=0.0 for now (baseline). We'll change to 0.3 later
# when we train the robust model.
train_dataset = SkinLesionDataset(
    dataframe=train_df,
    metadata_array=train_meta,
    transform=train_transform,
    modality_dropout_prob=0.0,
)

val_dataset = SkinLesionDataset(
    dataframe=val_df,
    metadata_array=val_meta,
    transform=eval_transform,
    modality_dropout_prob=0.0,
)

test_dataset = SkinLesionDataset(
    dataframe=test_df,
    metadata_array=test_meta,
    transform=eval_transform,
    modality_dropout_prob=0.0,
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 220
Val batches:   32
Test batches:  63


# **Sanity checks**

In [ ]:
# Pull one batch from the training loader
images, metas, labels = next(iter(train_loader))

print("=== Shape check ===")
print(f"Images shape: {images.shape}   (expected [{BATCH_SIZE}, 3, 224, 224])")
print(f"Metas shape:  {metas.shape}    (expected [{BATCH_SIZE}, {META_DIM}])")
print(f"Labels shape: {labels.shape}   (expected [{BATCH_SIZE}])")

# Manual asserts
if images.shape == (BATCH_SIZE, 3, 224, 224):
    print("Images shape: OK")
else:
    print("Images shape: WRONG")

if metas.shape == (BATCH_SIZE, META_DIM):
    print("Metas shape: OK")
else:
    print("Metas shape: WRONG")

if labels.shape == (BATCH_SIZE,):
    print("Labels shape: OK")
else:
    print("Labels shape: WRONG")

print("\n=== Range check ===")
print(f"Image min: {images.min():.3f}  max: {images.max():.3f}")
print(f"Meta min:  {metas.min():.3f}  max: {metas.max():.3f}")

# Get unique labels in this batch
unique_labels = []
for lbl in labels.tolist():
    if lbl not in unique_labels:
        unique_labels.append(lbl)
unique_labels.sort()
print(f"Unique labels in batch: {unique_labels}")

# === Modality Dropout check ===
print("\n=== Modality Dropout check ===")
dropout_check_ds = SkinLesionDataset(
    dataframe=train_df,
    metadata_array=train_meta,
    transform=eval_transform,
    modality_dropout_prob=0.30,
)

# Sample 500 items and count how many had metadata fully zeroed
n_total = 500
n_zeroed = 0
for i in range(n_total):
    _, m, _ = dropout_check_ds[i]
    # Check if every value in m is 0
    all_zero = True
    for value in m.tolist():
        if value != 0.0:
            all_zero = False
            break
    if all_zero:
        n_zeroed += 1

observed_rate = n_zeroed / n_total
print(f"Expected dropout rate: 0.30")
print(f"Observed dropout rate: {observed_rate:.3f}   ({n_zeroed}/{n_total} samples zeroed)")

if 0.20 < observed_rate < 0.40:
    print("Modality Dropout: OK")
else:
    print("Modality Dropout: RATE LOOKS OFF")

=== Shape check ===
Images shape: torch.Size([32, 3, 224, 224])   (expected [32, 3, 224, 224])
Metas shape:  torch.Size([32, 19])    (expected [32, 19])
Labels shape: torch.Size([32])   (expected [32])
Images shape: OK
Metas shape: OK
Labels shape: OK

=== Range check ===
Image min: -2.118  max: 2.640
Meta min:  0.000  max: 1.000
Unique labels in batch: [0, 1, 2, 3, 4]

=== Modality Dropout check ===
Expected dropout rate: 0.30
Observed dropout rate: 0.334   (167/500 samples zeroed)
Modality Dropout: OK


# **Phase -2**

# **Imports for the model**

In [ ]:
import torch.nn as nn
from torchvision import models

# **Vision Encoder (ResNet-18, partially frozen)**

In [ ]:
class VisionEncoder(nn.Module):
    """
    ResNet-18 pretrained on ImageNet.
    Early layers are frozen.
    The last 2 residual blocks (layer3 and layer4) are fine-tuned.
    Output: 512-dim feature vector per image.
    """

    def __init__(self):
        super().__init__()

        # Load pretrained ResNet-18
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        # Step 1: freeze every parameter first
        for param in resnet.parameters():
            param.requires_grad = False

        # Step 2: un-freeze the last 2 residual blocks (layer3 and layer4)
        for param in resnet.layer3.parameters():
            param.requires_grad = True
        for param in resnet.layer4.parameters():
            param.requires_grad = True

        # Step 3: remove the final classification layer (fc)
        # We keep everything up to and including avgpool, which outputs 512 features
        self.backbone = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
            resnet.avgpool,
        )

    def forward(self, x):
        # x shape: [batch, 3, 224, 224]
        features = self.backbone(x)
        # features shape after avgpool: [batch, 512, 1, 1]
        # Flatten to [batch, 512]
        features = features.view(features.size(0), -1)
        return features

# **Metadata Encoder (small MLP)**

In [ ]:
class MetadataEncoder(nn.Module):
    """
    Small MLP for tabular metadata.
    Input: 19-dim vector (1 age + 3 sex + 15 localization)
    Output: 128-dim feature vector.

    When the input vector is all zeros (because Modality Dropout zeroed it
    out, or because metadata is missing at inference), the output will be
    a fixed bias-dependent vector. That's intentional — the fusion layer
    learns to handle this case during training.
    """

    def __init__(self, input_dim):
        super().__init__()

        # Layer 1: 19 -> 64
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)

        # Layer 2: 64 -> 128
        self.fc2 = nn.Linear(64, 128)
        self.relu2 = nn.ReLU()

    def forward(self, x):
        # x shape: [batch, 19]
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        # output shape: [batch, 128]
        return x

# **Fusion + Classifier**

In [ ]:
class FusionClassifier(nn.Module):
    """
    Takes 512-dim image features and 128-dim metadata features,
    concatenates them into a 640-dim vector,
    passes through 2 FC layers, then outputs 7 class logits.
    """

    def __init__(self, image_dim, meta_dim, num_classes):
        super().__init__()

        fused_dim = image_dim + meta_dim  # 512 + 128 = 640

        # Block 1: 640 -> 256
        self.fc1 = nn.Linear(fused_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.4)

        # Block 2: 256 -> 128
        self.fc2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()

        # Output layer: 128 -> 7
        self.fc_out = nn.Linear(128, num_classes)

    def forward(self, image_features, meta_features):
        # image_features: [batch, 512]
        # meta_features:  [batch, 128]

        # Concatenate along feature dimension
        fused = torch.cat([image_features, meta_features], dim=1)
        # fused shape: [batch, 640]

        x = self.fc1(fused)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.relu2(x)

        logits = self.fc_out(x)
        # logits shape: [batch, 7]
        # Note: no softmax here — CrossEntropyLoss applies it internally
        return logits

# **The full model (composes the three components)**

In [ ]:
class MultimodalSkinLesionModel(nn.Module):
    """
    The full model: vision encoder + metadata encoder + fusion classifier.

    Note: Modality Dropout is NOT in this model — it lives in the Dataset.
    The model always receives a metadata vector; whether it's real or
    zeros is decided upstream.
    """

    def __init__(self, meta_input_dim, num_classes):
        super().__init__()

        self.vision_encoder = VisionEncoder()
        self.metadata_encoder = MetadataEncoder(input_dim=meta_input_dim)
        self.fusion_classifier = FusionClassifier(
            image_dim=512,
            meta_dim=128,
            num_classes=num_classes,
        )

    def forward(self, image, meta):
        # image: [batch, 3, 224, 224]
        # meta:  [batch, meta_input_dim]

        image_features = self.vision_encoder(image)
        meta_features = self.metadata_encoder(meta)
        logits = self.fusion_classifier(image_features, meta_features)

        return logits

# **Build the model and check it**

In [ ]:
# Pick the device
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"Device: {device}")

# Build the model
NUM_CLASSES = 7
model = MultimodalSkinLesionModel(
    meta_input_dim=META_DIM,
    num_classes=NUM_CLASSES,
)
model = model.to(device)

# Count parameters
total_params = 0
trainable_params = 0
for param in model.parameters():
    n = param.numel()
    total_params = total_params + n
    if param.requires_grad:
        trainable_params = trainable_params + n

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")

Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 190MB/s]


Total parameters:     11,384,519
Trainable parameters: 10,701,447
Frozen parameters:    683,072


 # **Forward-pass sanity check**

In [ ]:
# Pull one batch from the training loader
images, metas, labels = next(iter(train_loader))

# Move everything to the device
images = images.to(device)
metas = metas.to(device)
labels = labels.to(device)

print(f"Input images: {images.shape}")
print(f"Input metas:  {metas.shape}")
print(f"Labels:       {labels.shape}")

# Put model in eval mode so BatchNorm and Dropout don't fire during this check
model.eval()

# Forward pass with no gradient computation
with torch.no_grad():
    logits = model(images, metas)

print(f"\nOutput logits: {logits.shape}   (expected [32, 7])")
print(f"Logits min: {logits.min():.3f}  max: {logits.max():.3f}")

# Convert logits to predicted classes
predicted_classes = torch.argmax(logits, dim=1)
print(f"Predicted classes for this batch: {predicted_classes.tolist()}")
print(f"Actual labels for this batch:     {labels.tolist()}")

# Manual shape check
if logits.shape == (BATCH_SIZE, NUM_CLASSES):
    print("\nOutput shape: OK")
else:
    print("\nOutput shape: WRONG")

Input images: torch.Size([32, 3, 224, 224])
Input metas:  torch.Size([32, 19])
Labels:       torch.Size([32])

Output logits: torch.Size([32, 7])   (expected [32, 7])
Logits min: -0.336  max: 0.226
Predicted classes for this batch: [3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 6, 3, 4]
Actual labels for this batch:     [3, 1, 1, 1, 0, 4, 1, 1, 1, 4, 1, 0, 0, 1, 1, 1, 4, 1, 1, 0, 4, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1]

Output shape: OK


# **Test the missing-metadata path**

In [ ]:
# Take the same batch but zero out the metadata
metas_zero = torch.zeros_like(metas)

model.eval()
with torch.no_grad():
    logits_full = model(images, metas)        # with real metadata
    logits_zero = model(images, metas_zero)   # with zeroed metadata

print(f"Logits (full meta) shape: {logits_full.shape}")
print(f"Logits (zero meta) shape: {logits_zero.shape}")

# Compare predictions
pred_full = torch.argmax(logits_full, dim=1)
pred_zero = torch.argmax(logits_zero, dim=1)

n_same = 0
for i in range(len(pred_full)):
    if pred_full[i].item() == pred_zero[i].item():
        n_same = n_same + 1

print(f"\nPredictions matching between full and zero metadata: {n_same}/{len(pred_full)}")
print("(High match rate is expected for an untrained model — vision features dominate the random init)")

Logits (full meta) shape: torch.Size([32, 7])
Logits (zero meta) shape: torch.Size([32, 7])

Predictions matching between full and zero metadata: 32/32
(High match rate is expected for an untrained model — vision features dominate the random init)


# **Phase-3: Training**

# **Imports for training**

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
import time
import copy

# **Train-one-epoch function**

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    """
    Runs one full pass over the training loader.
    Returns: average loss, macro F1, accuracy
    """
    model.train()

    total_loss = 0.0
    n_batches = 0
    all_predictions = []
    all_labels = []

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for images, metas, labels in progress_bar:
        # Move to device
        images = images.to(device)
        metas = metas.to(device)
        labels = labels.to(device)

        # Forward pass
        logits = model(images, metas)
        loss = loss_fn(logits, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track loss
        total_loss = total_loss + loss.item()
        n_batches = n_batches + 1

        # Get predictions for metrics
        predicted_classes = torch.argmax(logits, dim=1)

        # Store on CPU as plain lists
        for p in predicted_classes.cpu().tolist():
            all_predictions.append(p)
        for l in labels.cpu().tolist():
            all_labels.append(l)

        # Update progress bar with running loss
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / n_batches
    macro_f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
    accuracy = accuracy_score(all_labels, all_predictions)

    return avg_loss, macro_f1, accuracy

# **Evaluate function**

In [ ]:
def evaluate(model, loader, loss_fn, device):
    """
    Runs one full pass over a val/test loader (no gradients).
    Returns: avg loss, macro F1, accuracy, predictions list, labels list
    """
    model.eval()

    total_loss = 0.0
    n_batches = 0
    all_predictions = []
    all_labels = []

    progress_bar = tqdm(loader, desc="Evaluating", leave=False)

    with torch.no_grad():
        for images, metas, labels in progress_bar:
            images = images.to(device)
            metas = metas.to(device)
            labels = labels.to(device)

            logits = model(images, metas)
            loss = loss_fn(logits, labels)

            total_loss = total_loss + loss.item()
            n_batches = n_batches + 1

            predicted_classes = torch.argmax(logits, dim=1)

            for p in predicted_classes.cpu().tolist():
                all_predictions.append(p)
            for l in labels.cpu().tolist():
                all_labels.append(l)

    avg_loss = total_loss / n_batches
    macro_f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
    accuracy = accuracy_score(all_labels, all_predictions)

    return avg_loss, macro_f1, accuracy, all_predictions, all_labels

# **Main training loop**

In [ ]:
def train_model(model, train_loader, val_loader, loss_fn, optimizer, scheduler,
                max_epochs, patience, device, checkpoint_path):
    """
    Trains the model with early stopping based on val Macro F1.
    Saves the best model state to checkpoint_path.
    Returns: history dict with per-epoch metrics
    """

    # History dict to store metrics each epoch
    history = {
        'train_loss': [],
        'train_f1': [],
        'train_acc': [],
        'val_loss': [],
        'val_f1': [],
        'val_acc': [],
        'lr': [],
    }

    best_val_f1 = 0.0
    best_epoch = 0
    epochs_without_improvement = 0

    start_time = time.time()

    for epoch in range(1, max_epochs + 1):
        epoch_start = time.time()

        print(f"\n--- Epoch {epoch}/{max_epochs} ---")

        # Train
        train_loss, train_f1, train_acc = train_one_epoch(
            model, train_loader, loss_fn, optimizer, device
        )

        # Validate
        val_loss, val_f1, val_acc, _, _ = evaluate(
            model, val_loader, loss_fn, device
        )

        # Get current learning rate (before scheduler updates it)
        current_lr = optimizer.param_groups[0]['lr']

        # Scheduler step (ReduceLROnPlateau watches val F1)
        scheduler.step(val_f1)

        # Record history
        history['train_loss'].append(train_loss)
        history['train_f1'].append(train_f1)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)

        epoch_time = time.time() - epoch_start

        # Print summary
        print(f"  Train  -- loss: {train_loss:.4f}  F1: {train_f1:.4f}  Acc: {train_acc:.4f}")
        print(f"  Val    -- loss: {val_loss:.4f}  F1: {val_f1:.4f}  Acc: {val_acc:.4f}")
        print(f"  LR: {current_lr:.6f}   Time: {epoch_time:.1f}s")

        # Check if this is the best model so far
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch
            epochs_without_improvement = 0

            # Save the best model
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_f1': val_f1,
                'val_acc': val_acc,
            }, checkpoint_path)
            print(f"  >> New best val F1: {val_f1:.4f}. Checkpoint saved.")
        else:
            epochs_without_improvement = epochs_without_improvement + 1
            print(f"  No improvement. ({epochs_without_improvement}/{patience})")

        # Early stopping
        if epochs_without_improvement >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch}.")
            print(f"Best val F1 was {best_val_f1:.4f} at epoch {best_epoch}.")
            break

    total_time = time.time() - start_time
    print(f"\n=== Training finished ===")
    print(f"Total time: {total_time/60:.1f} min")
    print(f"Best val F1: {best_val_f1:.4f} at epoch {best_epoch}")
    print(f"Best checkpoint saved to: {checkpoint_path}")

    return history

# **Smoke test (3 epochs, no Modality Dropout, no early stopping)**

In [ ]:
# Build a fresh model so we start clean
smoke_model = MultimodalSkinLesionModel(
    meta_input_dim=META_DIM,
    num_classes=NUM_CLASSES,
)
smoke_model = smoke_model.to(device)

# Weighted CrossEntropy with the class weights we computed earlier
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))

# Optimizer: only train parameters that require gradients
trainable_params_list = []
for param in smoke_model.parameters():
    if param.requires_grad:
        trainable_params_list.append(param)

smoke_optimizer = torch.optim.Adam(
    trainable_params_list,
    lr=1e-4,
    weight_decay=1e-5,
)

# Scheduler (won't really kick in during a 3-epoch test, but keeps the API consistent)
smoke_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    smoke_optimizer,
    mode='max',
    patience=5,
    factor=0.5,
)

print("=== SMOKE TEST: 3 epochs ===")
print("Looking for: train loss going DOWN and val F1 going UP across the 3 epochs.")
print()

smoke_history = train_model(
    model=smoke_model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=smoke_optimizer,
    scheduler=smoke_scheduler,
    max_epochs=3,
    patience=999,                        # disable early stopping for smoke test
    device=device,
    checkpoint_path='/kaggle/working/smoke_test.pt',
)

=== SMOKE TEST: 3 epochs ===
Looking for: train loss going DOWN and val F1 going UP across the 3 epochs.


--- Epoch 1/3 ---


  Train  -- loss: 1.5073  F1: 0.3273  Acc: 0.5039
  Val    -- loss: 1.1034  F1: 0.4782  Acc: 0.5858
  LR: 0.000100   Time: 79.1s
  >> New best val F1: 0.4782. Checkpoint saved.

--- Epoch 2/3 ---


  Train  -- loss: 1.0014  F1: 0.5113  Acc: 0.6541
  Val    -- loss: 0.8706  F1: 0.5797  Acc: 0.7265
  LR: 0.000100   Time: 49.8s
  >> New best val F1: 0.5797. Checkpoint saved.

--- Epoch 3/3 ---


  Train  -- loss: 0.8343  F1: 0.5659  Acc: 0.6902
  Val    -- loss: 0.8220  F1: 0.5834  Acc: 0.7066
  LR: 0.000100   Time: 51.0s
  >> New best val F1: 0.5834. Checkpoint saved.

=== Training finished ===
Total time: 3.0 min
Best val F1: 0.5834 at epoch 3
Best checkpoint saved to: /kaggle/working/smoke_test.pt


# **Full Baseline Training**

In [ ]:
# Build a fresh baseline model
baseline_model = MultimodalSkinLesionModel(
    meta_input_dim=META_DIM,
    num_classes=NUM_CLASSES,
)
baseline_model = baseline_model.to(device)

# Loss function (weighted CE)
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))

# Optimizer
trainable_params_list = []
for param in baseline_model.parameters():
    if param.requires_grad:
        trainable_params_list.append(param)

baseline_optimizer = torch.optim.Adam(
    trainable_params_list,
    lr=1e-4,
    weight_decay=1e-5,
)

# Scheduler
baseline_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    baseline_optimizer,
    mode='max',
    patience=5,
    factor=0.5,
)

# Confirm the training set is using NO modality dropout (this is the baseline)
print(f"Train dataset dropout prob: {train_dataset.dropout_prob}")
print("(should be 0.0 for the baseline)")
print()

print("=== BASELINE TRAINING: 50 epochs max, early stopping patience=10 ===")
print()

baseline_history = train_model(
    model=baseline_model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=baseline_optimizer,
    scheduler=baseline_scheduler,
    max_epochs=50,
    patience=10,
    device=device,
    checkpoint_path='/kaggle/working/baseline_best.pt',
)

Train dataset dropout prob: 0.0
(should be 0.0 for the baseline)

=== BASELINE TRAINING: 50 epochs max, early stopping patience=10 ===


--- Epoch 1/50 ---


  Train  -- loss: 1.5001  F1: 0.3250  Acc: 0.4368
  Val    -- loss: 1.1101  F1: 0.3864  Acc: 0.5459
  LR: 0.000100   Time: 50.3s
  >> New best val F1: 0.3864. Checkpoint saved.

--- Epoch 2/50 ---


  Train  -- loss: 0.9760  F1: 0.5107  Acc: 0.6625
  Val    -- loss: 0.8513  F1: 0.6347  Acc: 0.6916
  LR: 0.000100   Time: 50.9s
  >> New best val F1: 0.6347. Checkpoint saved.

--- Epoch 3/50 ---


  Train  -- loss: 0.8586  F1: 0.5517  Acc: 0.6796
  Val    -- loss: 0.8477  F1: 0.5297  Acc: 0.6238
  LR: 0.000100   Time: 51.5s
  No improvement. (1/10)

--- Epoch 4/50 ---


  Train  -- loss: 0.7696  F1: 0.5917  Acc: 0.7029
  Val    -- loss: 0.7853  F1: 0.5710  Acc: 0.6756
  LR: 0.000100   Time: 51.4s
  No improvement. (2/10)

--- Epoch 5/50 ---


...

  Train  -- loss: 0.1281  F1: 0.9200  Acc: 0.9234
  Val    -- loss: 0.9273  F1: 0.7578  Acc: 0.8373
  LR: 0.000025   Time: 50.3s
  No improvement. (6/10)

--- Epoch 43/50 ---


  Train  -- loss: 0.1192  F1: 0.9216  Acc: 0.9233
  Val    -- loss: 1.0067  F1: 0.7615  Acc: 0.8323
  LR: 0.000013   Time: 50.4s
  No improvement. (7/10)

--- Epoch 44/50 ---


  Train  -- loss: 0.1091  F1: 0.9283  Acc: 0.9265
  Val    -- loss: 0.9022  F1: 0.7684  Acc: 0.8353
  LR: 0.000013   Time: 50.0s
  No improvement. (8/10)

--- Epoch 45/50 ---


  Train  -- loss: 0.1073  F1: 0.9288  Acc: 0.9214
  Val    -- loss: 0.9432  F1: 0.7758  Acc: 0.8323
  LR: 0.000013   Time: 50.6s
  No improvement. (9/10)

--- Epoch 46/50 ---


  Train  -- loss: 0.1211  F1: 0.9304  Acc: 0.9281
  Val    -- loss: 0.9439  F1: 0.7691  Acc: 0.8253
  LR: 0.000013   Time: 50.8s
  No improvement. (10/10)

Early stopping triggered at epoch 46.
Best val F1 was 0.7807 at epoch 36.

=== Training finished ===
Total time: 38.9 min
Best val F1: 0.7807 at epoch 36
Best checkpoint saved to: /kaggle/working/baseline_best.pt


# **Save the baseline history to disk**

In [ ]:
import json

# Save history as JSON for easy loading later
with open('/kaggle/working/baseline_history.json', 'w') as f:
    json.dump(baseline_history, f, indent=2)

print("Baseline history saved to /kaggle/working/baseline_history.json")
print(f"Final epoch ran: {len(baseline_history['train_loss'])}")
print(f"Best val F1: {max(baseline_history['val_f1']):.4f}")

Baseline history saved to /kaggle/working/baseline_history.json
Final epoch ran: 46
Best val F1: 0.7807


# **Load the best baseline checkpoint and evaluate at 4 missing rates**

In [ ]:
# Reload the best baseline checkpoint
checkpoint = torch.load('/kaggle/working/baseline_best.pt', weights_only=False)
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"Saved val F1: {checkpoint['val_f1']:.4f}")
print(f"Saved val Acc: {checkpoint['val_acc']:.4f}")

# Build a fresh model and load the weights
eval_model = MultimodalSkinLesionModel(
    meta_input_dim=META_DIM,
    num_classes=NUM_CLASSES,
)
eval_model.load_state_dict(checkpoint['model_state_dict'])
eval_model = eval_model.to(device)
eval_model.eval()

# Loss function (same as training, for comparable loss numbers)
eval_loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))

# The 4 missing-metadata conditions
missing_rates = [0.0, 0.25, 0.50, 1.0]

# Results dict
baseline_results = {}

for rate in missing_rates:
    print(f"\n=== Evaluating baseline at {int(rate*100)}% missing metadata ===")

    # Build a test dataset with this dropout rate
    test_dataset_at_rate = SkinLesionDataset(
        dataframe=test_df,
        metadata_array=test_meta,
        transform=eval_transform,
        modality_dropout_prob=rate,
    )
    test_loader_at_rate = DataLoader(
        test_dataset_at_rate,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    # Evaluate
    loss_val, f1_val, acc_val, preds, labels = evaluate(
        eval_model, test_loader_at_rate, eval_loss_fn, device
    )

    # Per-class precision/recall/F1
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)

    # Confusion matrix
    cm = confusion_matrix(labels, preds, labels=[0, 1, 2, 3, 4, 5, 6])

    # Store everything
    baseline_results[rate] = {
        'loss': loss_val,
        'macro_f1': f1_val,
        'accuracy': acc_val,
        'macro_precision': precision,
        'macro_recall': recall,
        'predictions': preds,
        'labels': labels,
        'confusion_matrix': cm.tolist(),
    }

    print(f"  Loss:            {loss_val:.4f}")
    print(f"  Macro F1:        {f1_val:.4f}")
    print(f"  Accuracy:        {acc_val:.4f}")
    print(f"  Macro Precision: {precision:.4f}")
    print(f"  Macro Recall:    {recall:.4f}")

print("\n=== BASELINE DECAY SUMMARY ===")
print(f"{'Missing':<12}{'Loss':>10}{'Macro F1':>12}{'Accuracy':>12}")
for rate in missing_rates:
    r = baseline_results[rate]
    print(f"{int(rate*100):>3}%        {r['loss']:>10.4f}{r['macro_f1']:>12.4f}{r['accuracy']:>12.4f}")

Loaded checkpoint from epoch 36
Saved val F1: 0.7807
Saved val Acc: 0.8194

=== Evaluating baseline at 0% missing metadata ===


  Loss:            0.7788
  Macro F1:        0.7490
  Accuracy:        0.8248
  Macro Precision: 0.7174
  Macro Recall:    0.7953

=== Evaluating baseline at 25% missing metadata ===


  Loss:            0.7897
  Macro F1:        0.7477
  Accuracy:        0.8268
  Macro Precision: 0.7160
  Macro Recall:    0.7944

=== Evaluating baseline at 50% missing metadata ===


  Loss:            0.8042
  Macro F1:        0.7478
  Accuracy:        0.8298
  Macro Precision: 0.7211
  Macro Recall:    0.7878

=== Evaluating baseline at 100% missing metadata ===


  Loss:            0.8641
  Macro F1:        0.7489
  Accuracy:        0.8303
  Macro Precision: 0.7310
  Macro Recall:    0.7803

=== BASELINE DECAY SUMMARY ===
Missing           Loss    Macro F1    Accuracy
  0%            0.7788      0.7490      0.8248
 25%            0.7897      0.7477      0.8268
 50%            0.8042      0.7478      0.8298
100%            0.8641      0.7489      0.8303


In [ ]:
class MetadataOnlyModel(nn.Module):
    """
    Just the metadata MLP followed by a classifier.
    No images involved. Used to diagnose how much signal
    lives in (age, sex, localization) alone.
    """

    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(64, 128)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)

        self.fc_out = nn.Linear(128, num_classes)

    def forward(self, meta):
        x = self.fc1(meta)
        x = self.relu1(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)

        logits = self.fc_out(x)
        return logits

In [ ]:
def train_one_epoch_meta_only(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0.0
    n_batches = 0
    all_predictions = []
    all_labels = []

    for images, metas, labels in loader:
        # We ignore images entirely
        metas = metas.to(device)
        labels = labels.to(device)

        logits = model(metas)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss = total_loss + loss.item()
        n_batches = n_batches + 1

        predicted_classes = torch.argmax(logits, dim=1)
        for p in predicted_classes.cpu().tolist():
            all_predictions.append(p)
        for l in labels.cpu().tolist():
            all_labels.append(l)

    avg_loss = total_loss / n_batches
    macro_f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
    accuracy = accuracy_score(all_labels, all_predictions)
    return avg_loss, macro_f1, accuracy


def evaluate_meta_only(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for images, metas, labels in loader:
            metas = metas.to(device)
            labels = labels.to(device)

            logits = model(metas)
            loss = loss_fn(logits, labels)

            total_loss = total_loss + loss.item()
            n_batches = n_batches + 1

            predicted_classes = torch.argmax(logits, dim=1)
            for p in predicted_classes.cpu().tolist():
                all_predictions.append(p)
            for l in labels.cpu().tolist():
                all_labels.append(l)

    avg_loss = total_loss / n_batches
    macro_f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
    accuracy = accuracy_score(all_labels, all_predictions)
    return avg_loss, macro_f1, accuracy

In [ ]:
# Build the metadata-only model
meta_model = MetadataOnlyModel(input_dim=META_DIM, num_classes=NUM_CLASSES)
meta_model = meta_model.to(device)

# Same loss & optimizer setup as before
meta_loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
meta_optimizer = torch.optim.Adam(meta_model.parameters(), lr=1e-3, weight_decay=1e-5)

MAX_EPOCHS_META = 30

best_val_f1 = 0.0
best_state = None

print("=== Training metadata-only diagnostic model ===")
print()

for epoch in range(1, MAX_EPOCHS_META + 1):
    train_loss, train_f1, train_acc = train_one_epoch_meta_only(
        meta_model, train_loader, meta_loss_fn, meta_optimizer, device
    )
    val_loss, val_f1, val_acc = evaluate_meta_only(
        meta_model, val_loader, meta_loss_fn, device
    )

    flag = ""
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = copy.deepcopy(meta_model.state_dict())
        flag = " <-- best"

    print(f"Epoch {epoch:2d}  train F1: {train_f1:.4f}  val F1: {val_f1:.4f}  val Acc: {val_acc:.4f}{flag}")

# Restore best weights
meta_model.load_state_dict(best_state)
print(f"\nBest val F1 (metadata-only): {best_val_f1:.4f}")

=== Training metadata-only diagnostic model ===

Epoch  1  train F1: 0.1770  val F1: 0.2055  val Acc: 0.4711 <-- best
Epoch  2  train F1: 0.2115  val F1: 0.2308  val Acc: 0.4391 <-- best
Epoch  3  train F1: 0.2142  val F1: 0.1846  val Acc: 0.3683
Epoch  4  train F1: 0.2119  val F1: 0.1838  val Acc: 0.3473
Epoch  5  train F1: 0.2188  val F1: 0.2305  val Acc: 0.4551
Epoch  6  train F1: 0.2188  val F1: 0.2145  val Acc: 0.3952
Epoch  7  train F1: 0.2186  val F1: 0.2057  val Acc: 0.4112
Epoch  8  train F1: 0.2230  val F1: 0.1944  val Acc: 0.3782
Epoch  9  train F1: 0.2234  val F1: 0.1942  val Acc: 0.3683
Epoch 10  train F1: 0.2266  val F1: 0.2158  val Acc: 0.4182
Epoch 11  train F1: 0.2250  val F1: 0.2334  val Acc: 0.4551 <-- best
Epoch 12  train F1: 0.2257  val F1: 0.2176  val Acc: 0.4002
Epoch 13  train F1: 0.2214  val F1: 0.2035  val Acc: 0.4012
Epoch 14  train F1: 0.2165  val F1: 0.2203  val Acc: 0.3852
Epoch 15  train F1: 0.2238  val F1: 0.2269  val Acc: 0.4062
Epoch 16  train F1: 0.22

In [ ]:
test_loss, test_f1, test_acc = evaluate_meta_only(
    meta_model, test_loader, meta_loss_fn, device
)

print("=== METADATA-ONLY DIAGNOSTIC RESULT ===")
print(f"Test Macro F1: {test_f1:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print()
print("Reference points:")
print(f"  Random (uniform):     ~0.143")
print(f"  Majority class (NV):  ~0.115  (macro F1, not accuracy)")
print(f"  Multimodal baseline:  0.7490  (test Macro F1, your full model)")
print(f"  Image-only target:    we'll measure this next if needed")

=== METADATA-ONLY DIAGNOSTIC RESULT ===
Test Macro F1: 0.2416
Test Accuracy: 0.4838

Reference points:
  Random (uniform):     ~0.143
  Majority class (NV):  ~0.115  (macro F1, not accuracy)
  Multimodal baseline:  0.7490  (test Macro F1, your full model)
  Image-only target:    we'll measure this next if needed


In [ ]:
class ImageOnlyModel(nn.Module):
    """
    Just the vision encoder + a classifier. No metadata at all.
    Used to measure the image-only ceiling.
    """

    def __init__(self, num_classes):
        super().__init__()
        self.vision_encoder = VisionEncoder()
        # Same classifier head depth as multimodal, but input is 512 not 640
        self.fc1 = nn.Linear(512, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()

        self.fc_out = nn.Linear(128, num_classes)

    def forward(self, image):
        x = self.vision_encoder(image)
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        logits = self.fc_out(x)
        return logits

In [ ]:
def train_one_epoch_img_only(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0.0
    n_batches = 0
    all_predictions = []
    all_labels = []

    progress_bar = tqdm(loader, desc="Training (img-only)", leave=False)

    for images, metas, labels in progress_bar:
        # Ignore metas entirely
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss = total_loss + loss.item()
        n_batches = n_batches + 1

        predicted_classes = torch.argmax(logits, dim=1)
        for p in predicted_classes.cpu().tolist():
            all_predictions.append(p)
        for l in labels.cpu().tolist():
            all_labels.append(l)

        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / n_batches
    macro_f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
    accuracy = accuracy_score(all_labels, all_predictions)
    return avg_loss, macro_f1, accuracy


def evaluate_img_only(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for images, metas, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = loss_fn(logits, labels)

            total_loss = total_loss + loss.item()
            n_batches = n_batches + 1

            predicted_classes = torch.argmax(logits, dim=1)
            for p in predicted_classes.cpu().tolist():
                all_predictions.append(p)
            for l in labels.cpu().tolist():
                all_labels.append(l)

    avg_loss = total_loss / n_batches
    macro_f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
    accuracy = accuracy_score(all_labels, all_predictions)
    return avg_loss, macro_f1, accuracy

In [ ]:
img_model = ImageOnlyModel(num_classes=NUM_CLASSES)
img_model = img_model.to(device)

img_loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))

# Only train parameters that require gradients (layer3, layer4, head)
img_trainable_params = []
for param in img_model.parameters():
    if param.requires_grad:
        img_trainable_params.append(param)

img_optimizer = torch.optim.Adam(img_trainable_params, lr=1e-4, weight_decay=1e-5)
img_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(img_optimizer, mode='max', patience=3, factor=0.5)

MAX_EPOCHS_IMG = 20
PATIENCE_IMG = 5

best_val_f1 = 0.0
best_state = None
epochs_without_improvement = 0

print("=== Training image-only diagnostic model ===")
print()

for epoch in range(1, MAX_EPOCHS_IMG + 1):
    epoch_start = time.time()

    train_loss, train_f1, train_acc = train_one_epoch_img_only(
        img_model, train_loader, img_loss_fn, img_optimizer, device
    )
    val_loss, val_f1, val_acc = evaluate_img_only(
        img_model, val_loader, img_loss_fn, device
    )

    img_scheduler.step(val_f1)

    flag = ""
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = copy.deepcopy(img_model.state_dict())
        epochs_without_improvement = 0
        flag = " <-- best"
    else:
        epochs_without_improvement = epochs_without_improvement + 1

    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch:2d}  train F1: {train_f1:.4f}  val F1: {val_f1:.4f}  val Acc: {val_acc:.4f}  ({epoch_time:.0f}s){flag}")

    if epochs_without_improvement >= PATIENCE_IMG:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

img_model.load_state_dict(best_state)
print(f"\nBest val F1 (image-only): {best_val_f1:.4f}")

=== Training image-only diagnostic model ===



Epoch  1  train F1: 0.3420  val F1: 0.5126  val Acc: 0.6138  (51s) <-- best


Epoch  2  train F1: 0.4974  val F1: 0.4431  val Acc: 0.5719  (51s)


Epoch  3  train F1: 0.5451  val F1: 0.5697  val Acc: 0.6557  (51s) <-- best


Epoch  4  train F1: 0.5891  val F1: 0.5388  val Acc: 0.7226  (51s)


...

Epoch 16  train F1: 0.7694  val F1: 0.6902  val Acc: 0.8074  (53s)


Epoch 17  train F1: 0.7321  val F1: 0.6568  val Acc: 0.7515  (52s)


Epoch 18  train F1: 0.8092  val F1: 0.7282  val Acc: 0.8164  (52s) <-- best


Epoch 19  train F1: 0.8059  val F1: 0.7078  val Acc: 0.7784  (52s)


Epoch 20  train F1: 0.7958  val F1: 0.7254  val Acc: 0.7894  (52s)

Best val F1 (image-only): 0.7282


In [ ]:
test_loss, test_f1, test_acc = evaluate_img_only(
    img_model, test_loader, img_loss_fn, device
)

print("=== IMAGE-ONLY DIAGNOSTIC RESULT ===")
print(f"Test Macro F1: {test_f1:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print()
print("=== FULL COMPARISON ===")
print(f"{'Model':<35}{'Test Macro F1':>15}")
print(f"{'Random':<35}{'~0.143':>15}")
print(f"{'Metadata only':<35}{0.2416:>15.4f}")
print(f"{'Image only':<35}{test_f1:>15.4f}")
print(f"{'Multimodal baseline':<35}{0.7490:>15.4f}")
print()
print(f"Gap from image-only to multimodal: {0.7490 - test_f1:+.4f}")
print()
print("Interpretation:")
print("  Gap > 0.05  -> metadata is meaningfully contributing, rebalancing has room to work")
print("  Gap 0.02-0.05 -> small but real contribution, paper is harder but possible")
print("  Gap < 0.02  -> images dominate completely, project needs reframing")

=== IMAGE-ONLY DIAGNOSTIC RESULT ===
Test Macro F1: 0.7204
Test Accuracy: 0.8183

=== FULL COMPARISON ===
Model                                Test Macro F1
Random                                      ~0.143
Metadata only                               0.2416
Image only                                  0.7204
Multimodal baseline                         0.7490

Gap from image-only to multimodal: +0.0286

Interpretation:
  Gap > 0.05  -> metadata is meaningfully contributing, rebalancing has room to work
  Gap 0.02-0.05 -> small but real contribution, paper is harder but possible
  Gap < 0.02  -> images dominate completely, project needs reframing
